# MODIS → VIIRS : نموذج تقدير (Colab)

الموديل بيتدرّب على سنتين فيهما **MODIS + VIIRS** (مثلًا 2025)، ويتعلم من خصائص MODIS
(عدد الـ detections، FRP، brightness temperatures، confidence، التوقيت، الجيران، الموسم)
إنه يتنبأ بـ **VIIRS** لنفس الخلية واليوم على شبكة 0.05°.

بعد التدريب بنطبّقه على سنين **MODIS فقط** (مثل 2005) فنطلع **تقدير VIIRS-equivalent**.

> **مهم:** VIIRS لم يكن موجودًا قبل 2012. المخرجات لسنوات 2000–2012 ليست أرصادًا حقيقية،
> بل تقديرات موديل، وكل صف فيها معلم `is_estimate = True`.

**قبل التشغيل:** لازم يكون معك `MAP_KEY` مجاني من https://firms.modaps.eosdis.nasa.gov/api/map_key/


In [ ]:
!pip -q install lightgbm xgboost pyarrow

import os, json, time, io, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import joblib
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
                             f1_score, confusion_matrix, roc_curve, mean_absolute_error,
                             mean_squared_error, r2_score)
import lightgbm as lgb
import xgboost as xgb
warnings.filterwarnings("ignore")
print("ready")

In [ ]:
# ----------------------------- الإعدادات -----------------------------
MAP_KEY = "PUT_YOUR_FIRMS_MAP_KEY"          # من https://firms.modaps.eosdis.nasa.gov/api/map_key/
BBOX = (0.7, 6.2, 3.9, 12.5)                 # west, south, east, north  (بنين كمثال)
TRAIN_YEARS = [2025]                         # السنة اللي فيها MODIS + VIIRS
PREDICT_YEARS = [2005]                       # السنة اللي هنقدّر لها VIIRS
SOURCES = {"modis": "MODIS_SP", "viirs": "VIIRS_SNPP_SP"}
GRID = 0.05
DRIVE = "/content/drive/MyDrive/pyroscope_model"
os.makedirs(f"{DRIVE}/raw", exist_ok=True)

from google.colab import drive
drive.mount("/content/drive")
print(DRIVE)

## 1) تنزيل الداتا من FIRMS

الـ Area API بترجع بحد أقصى 5 أيام في الطلب، فبنقسّم السنة لأجزاء ونخزّن الناتج على Drive
عشان ما ننزّلش تاني لو الـ runtime اتقطع.

In [ ]:
def download_year(source: str, year: int, bbox=BBOX, key=MAP_KEY) -> pd.DataFrame:
    cache = Path(DRIVE) / "raw" / f"{source}_{year}_{bbox[0]}_{bbox[1]}_{bbox[2]}_{bbox[3]}.csv"
    if cache.exists():
        print("cached:", cache.name)
        return pd.read_csv(cache, low_memory=False)

    west, south, east, north = bbox
    area = f"{west},{south},{east},{north}"
    frames, day = [], pd.Timestamp(year=year, month=1, day=1)
    end = pd.Timestamp(year=year, month=12, day=31)
    while day <= end:
        chunk_end = min(day + pd.Timedelta(days=4), end)
        n_days = (chunk_end - day).days + 1
        url = (f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{key}/{source}/"
               f"{area}/{n_days}/{day.date()}")
        response = requests.get(url, timeout=180)
        text = response.text.strip()
        if response.status_code == 200 and text and "Invalid" not in text[:20] and "," in text.split("\n")[0]:
            frames.append(pd.read_csv(io.StringIO(text)))
        else:
            print(f"  {day.date()} -> {text[:80]}")
        day = chunk_end + pd.Timedelta(days=1)
        time.sleep(0.2)

    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    df.to_csv(cache, index=False)
    print(f"{source} {year}: {len(df):,} rows -> {cache.name}")
    return df

raw = {}
for year in TRAIN_YEARS:
    for sensor, source in SOURCES.items():
        raw[(sensor, year)] = download_year(source, year)
for year in PREDICT_YEARS:
    raw[("modis", year)] = download_year(SOURCES["modis"], year)

## 2) التنظيف وبناء جدول cell-day (0.05°)

قواعد التنظيف: استبعاد `type != 0` (حرائق غير نباتية)، إسقاط الإحداثيات/التوقيت غير الصالح،
حذف التكرار الحقيقي، وإسقاط FRP السالب أو المستحيل. الـ confidence عند MODIS رقمي (0–100) وبيتحول
لفئات `low/nominal/high`، وعند VIIRS هو أصلًا `l/n/h`.

In [ ]:
MOD_COLS = {"latitude":"lat","longitude":"lon","brightness":"bt_primary","bright_t31":"bt_secondary",
            "scan":"scan","track":"track","acq_date":"date","acq_time":"time","satellite":"platform",
            "confidence":"confidence_raw","frp":"frp","daynight":"daynight","type":"type"}
VII_COLS = {"latitude":"lat","longitude":"lon","bright_ti4":"bt_primary","bright_ti5":"bt_secondary",
            "scan":"scan","track":"track","acq_date":"date","acq_time":"time","satellite":"platform",
            "confidence":"confidence_raw","frp":"frp","daynight":"daynight","type":"type"}

def clean(raw_df, sensor):
    df = raw_df[list((MOD_COLS if sensor=="modis" else VII_COLS))].rename(
        columns=MOD_COLS if sensor=="modis" else VII_COLS).copy()
    for c in ["lat","lon","frp","bt_primary","bt_secondary","scan","track","type"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    if sensor == "viirs":
        conf = df["confidence_raw"].astype(str).str.strip().str.lower().replace({"low":"l","nominal":"n","high":"h"})
        df["confidence_class"] = conf.map({"l":"low","n":"nominal","h":"high"})
    else:
        df["confidence_class"] = pd.cut(df["confidence_raw"].astype(float), [-1,30,80,101],
                                        labels=["low","nominal","high"]).astype("object")
    t = df["time"].fillna(0).astype(int).clip(0,2359).astype(str).str.zfill(4)
    df["datetime_utc"] = pd.to_datetime(df["date"].astype(str)+" "+t.str[:2]+":"+t.str[2:],
                                        format="%Y-%m-%d %H:%M", errors="coerce")
    df["hour"] = df["datetime_utc"].dt.hour + df["datetime_utc"].dt.minute/60
    df = df[df["type"].fillna(0) == 0]
    df = df[df["lat"].between(-90,90) & df["lon"].between(-180,180) & df["datetime_utc"].notna()]
    df = df.drop_duplicates(subset=["lat","lon","datetime_utc","platform","frp","bt_primary"])
    df = df[df["frp"].between(0, 10_000)]
    df["date"] = df["datetime_utc"].dt.normalize()
    df["cell_row"] = np.floor(df["lat"]/GRID).astype(int)
    df["cell_col"] = np.floor(df["lon"]/GRID).astype(int)
    df["cell_id"] = df["cell_row"].map(lambda r: f"r{r:04d}") + df["cell_col"].map(lambda c: f"c{c:04d}")
    return df

def aggregate(df, sensor):
    g = df.groupby(["cell_id","cell_row","cell_col","date"], observed=True)
    agg = g.agg(detections=("frp","size"), frp_mean=("frp","mean"), frp_max=("frp","max"),
                frp_sum=("frp","sum"), frp_std=("frp","std"), bt_primary_mean=("bt_primary","mean"),
                bt_primary_max=("bt_primary","max"), bt_secondary_mean=("bt_secondary","mean"),
                bt_secondary_max=("bt_secondary","max"), scan_mean=("scan","mean"), track_mean=("track","mean"),
                hour_mean=("hour","mean"), night_share=("daynight", lambda s: float((s=="N").mean())),
                platforms=("platform","nunique")).reset_index()
    conf = (pd.crosstab([df["cell_id"], df["date"]], df["confidence_class"])
            .reindex(columns=["low","nominal","high"], fill_value=0).reset_index()
            .rename(columns={"low":"conf_low","nominal":"conf_nominal","high":"conf_high"}))
    agg = agg.merge(conf, on=["cell_id","date"], how="left")
    keep = ("cell_id","cell_row","cell_col","date")
    return agg.rename(columns={c: f"{sensor}_{c}" for c in agg.columns if c not in keep})

def add_context(cell_days):
    df = cell_days.sort_values("date").copy()
    lookup = df.set_index(["cell_row","cell_col","date"])["modis_detections"]
    neighbours = []
    for dr in (-1,0,1):
        for dc in (-1,0,1):
            if dr == 0 and dc == 0: continue
            idx = pd.MultiIndex.from_arrays([df["cell_row"]+dr, df["cell_col"]+dc, df["date"]])
            neighbours.append(pd.Series(lookup.reindex(idx).to_numpy(), index=df.index).fillna(0))
    df["modis_neighbour_count"] = np.sum(neighbours, axis=0)
    counts = df.set_index(["cell_id","date"])["modis_detections"]
    for lag in (1,2,3):
        idx = pd.MultiIndex.from_arrays([df["cell_id"], df["date"] - pd.Timedelta(days=lag)])
        df[f"modis_prev{lag}_count"] = pd.Series(counts.reindex(idx).to_numpy(), index=df.index).fillna(0)
    df["modis_prev3_mean"] = df[[f"modis_prev{i}_count" for i in (1,2,3)]].mean(axis=1)
    df["cell_lat"] = (df["cell_row"]+0.5)*GRID
    df["cell_lon"] = (df["cell_col"]+0.5)*GRID
    df["doy"] = df["date"].dt.day_of_year
    df["doy_sin"] = np.sin(2*np.pi*df["doy"]/365.25); df["doy_cos"] = np.cos(2*np.pi*df["doy"]/365.25)
    df["hour_sin"] = np.sin(2*np.pi*df["modis_hour_mean"]/24); df["hour_cos"] = np.cos(2*np.pi*df["modis_hour_mean"]/24)
    return df

def build_train_table(modis_raw, viirs_raw, year):
    modis_cd = aggregate(clean(modis_raw, "modis"), "modis")
    viirs_cd = aggregate(clean(viirs_raw, "viirs"), "viirs").drop(columns=["cell_row","cell_col"])
    table = modis_cd.merge(viirs_cd, on=["cell_id","date"], how="left")
    targets = [c for c in table.columns if c.startswith("viirs_")]
    table[targets] = table[targets].fillna(0)
    table = add_context(table); table["year"] = year
    print(f"{year}: {len(table):,} MODIS cell-days, positive rate {(table.viirs_detections>0).mean():.3f}, "
          f"VIIRS-only cell-days excluded: {len(viirs_cd) - len(table):,}")
    return table

tables = [build_train_table(raw[("modis", y)], raw[("viirs", y)], y) for y in TRAIN_YEARS]
df = pd.concat(tables, ignore_index=True)
print("total rows:", len(df))

## 3) المقارنة: normalisation vs standardisation + اختيار أحسن موديل

بنقسّم **80/20 باليوم** (GroupShuffleSplit على التاريخ) عشان خلايا نفس اليوم متبقاش في التدريب والاختبار مع بعض.
وبنقارن 4 معالجات × 5 موديلات تصنيف + 5 موديلات انحدار.

In [ ]:
FEATURES = ["modis_detections","modis_frp_mean","modis_frp_max","modis_frp_sum","modis_frp_std",
            "modis_bt_primary_mean","modis_bt_primary_max","modis_bt_secondary_mean","modis_bt_secondary_max",
            "modis_scan_mean","modis_track_mean","modis_hour_mean","modis_night_share","modis_platforms",
            "modis_conf_low","modis_conf_nominal","modis_conf_high","modis_neighbour_count",
            "modis_prev1_count","modis_prev2_count","modis_prev3_count","modis_prev3_mean",
            "doy_sin","doy_cos","hour_sin","hour_cos","cell_lat","cell_lon"]
SKEWED = [f for f in FEATURES if f.endswith(("_detections","_count","_mean","_max","_sum","_std"))]

def prep(kind):
    if kind == "none": return ColumnTransformer([("keep","passthrough",FEATURES)])
    if kind == "standard": return ColumnTransformer([("s", Pipeline([("i",SimpleImputer(strategy="median")),("sc",StandardScaler())]), FEATURES)])
    if kind == "minmax":   return ColumnTransformer([("s", Pipeline([("i",SimpleImputer(strategy="median")),("sc",MinMaxScaler())]), FEATURES)])
    skewed = [f for f in SKEWED if f in FEATURES]; rest = [f for f in FEATURES if f not in skewed]
    return ColumnTransformer([("log", Pipeline([("i",SimpleImputer(strategy="median")),("l",FunctionTransformer(np.log1p)),("sc",StandardScaler())]), skewed),
                              ("plain", Pipeline([("i",SimpleImputer(strategy="median")),("sc",StandardScaler())]), rest)])

def classifiers(kind):
    return {"logistic_regression": Pipeline([("p",prep("standard" if kind=="none" else kind)),("m",LogisticRegression(max_iter=2000))]),
            "random_forest": Pipeline([("p",prep(kind)),("m",RandomForestClassifier(n_estimators=300,min_samples_leaf=5,n_jobs=-1,random_state=0))]),
            "hist_gradient_boosting": Pipeline([("p",prep(kind)),("m",HistGradientBoostingClassifier(max_iter=300,learning_rate=0.08,random_state=0))]),
            "lightgbm": Pipeline([("p",prep(kind)),("m",lgb.LGBMClassifier(n_estimators=500,learning_rate=0.05,num_leaves=31,verbose=-1,random_state=0))]),
            "xgboost": Pipeline([("p",prep(kind)),("m",xgb.XGBClassifier(n_estimators=500,learning_rate=0.05,max_depth=6,subsample=0.9,colsample_bytree=0.9,eval_metric="logloss",random_state=0))])}

def regressors(kind):
    return {"poisson_regression": Pipeline([("p",prep("standard" if kind=="none" else kind)),("m",PoissonRegressor(max_iter=500))]),
            "random_forest": Pipeline([("p",prep(kind)),("m",RandomForestRegressor(n_estimators=300,min_samples_leaf=3,n_jobs=-1,random_state=0))]),
            "hist_gradient_boosting": Pipeline([("p",prep(kind)),("m",HistGradientBoostingRegressor(loss="poisson",max_iter=300,learning_rate=0.08,random_state=0))]),
            "lightgbm": Pipeline([("p",prep(kind)),("m",lgb.LGBMRegressor(objective="poisson",n_estimators=500,learning_rate=0.05,num_leaves=31,verbose=-1,random_state=0))]),
            "xgboost": Pipeline([("p",prep(kind)),("m",xgb.XGBRegressor(objective="count:poisson",n_estimators=500,learning_rate=0.05,max_depth=6,subsample=0.9,colsample_bytree=0.9,random_state=0))])}

X = df[FEATURES]
y_cls = (df["viirs_detections"] > 0).astype(int).to_numpy()
y_cnt = df["viirs_detections"].to_numpy(float)
groups = df["date"].dt.strftime("%Y-%m-%d")
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0).split(X, y_cls, groups))
print(f"train {len(tr):,} / test {len(te):,}")

results = {"classification": {}, "regression": {}}
best_cls, best_reg = (-1, None, None), (1e9, None, None)
for kind in ["none","standard","minmax","log1p_standard"]:
    for name, model in classifiers(kind).items():
        model.fit(X.iloc[tr], y_cls[tr])
        proba = model.predict_proba(X.iloc[te])[:,1]
        p, r, th = precision_recall_curve(y_cls[te], proba)
        f1 = np.divide(2*p*r, p+r, out=np.zeros_like(p), where=(p+r)>0)
        i = int(np.nanargmax(f1[:-1]))
        key = f"{name}__{kind}"
        results["classification"][key] = {"roc_auc": roc_auc_score(y_cls[te], proba), "pr_auc": average_precision_score(y_cls[te], proba),
                                          "f1": float(f1[i]), "threshold": float(th[i]), "precision": float(p[i]), "recall": float(r[i])}
        print(f"[cls] {key:42s} ROC={results['classification'][key]['roc_auc']:.4f} PR={results['classification'][key]['pr_auc']:.4f} F1={f1[i]:.4f}")
        if results["classification"][key]["pr_auc"] > best_cls[0]: best_cls = (results["classification"][key]["pr_auc"], key, model)
    for name, model in regressors(kind).items():
        model.fit(X.iloc[tr], y_cnt[tr])
        pred = np.clip(model.predict(X.iloc[te]), 0, None)
        key = f"{name}__{kind}"
        results["regression"][key] = {"mae": mean_absolute_error(y_cnt[te], pred), "rmse": float(np.sqrt(mean_squared_error(y_cnt[te], pred))),
                                      "r2": r2_score(y_cnt[te], pred), "spearman": float(pd.Series(y_cnt[te]).corr(pd.Series(pred), method="spearman"))}
        print(f"[reg] {key:42s} MAE={results['regression'][key]['mae']:.3f} R2={results['regression'][key]['r2']:.3f}")
        if results["regression"][key]["mae"] < best_reg[0]: best_reg = (results["regression"][key]["mae"], key, model)

print("\nbest:", best_cls[1], "|", best_reg[1])

## 4) Confusion matrix + ROC للموديل الأفضل

In [ ]:
proba = best_cls[2].predict_proba(X.iloc[te])[:,1]
threshold = results["classification"][best_cls[1]]["threshold"]
cm = confusion_matrix(y_cls[te], (proba >= threshold).astype(int), labels=[0,1])
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].imshow(cm, cmap="Oranges")
ax[0].set_xticks([0,1], ["pred: no VIIRS","pred: VIIRS"]); ax[0].set_yticks([0,1], ["true: no VIIRS","true: VIIRS"])
for i in range(2):
    for j in range(2):
        ax[0].text(j, i, f"{cm[i,j]:,}", ha="center", va="center", fontsize=12)
ax[0].set_title(f"Confusion matrix @ {threshold:.2f}  F1={f1_score(y_cls[te],(proba>=threshold).astype(int)):.3f}")
fpr, tpr, _ = roc_curve(y_cls[te], proba)
ax[1].plot(fpr, tpr, lw=2, color="#ea580c"); ax[1].plot([0,1],[0,1],"--",color="grey")
ax[1].set_xlabel("false positive rate"); ax[1].set_ylabel("true positive rate"); ax[1].set_title(f"ROC-AUC={results['classification'][best_cls[1]]['roc_auc']:.3f}")
plt.tight_layout(); plt.savefig(f"{DRIVE}/confusion_matrix.png", dpi=150); plt.show()
print(cm)

## 5) حفظ الموديلات على Drive

In [ ]:
frp_mask = y_cnt[tr] > 0
frp_model = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, verbose=-1, random_state=0)
frp_model.fit(X.iloc[tr][frp_mask], np.log1p(df["viirs_frp_mean"].to_numpy()[tr][frp_mask]))

joblib.dump(best_cls[2], f"{DRIVE}/model_presence.joblib")
joblib.dump(best_reg[2], f"{DRIVE}/model_count.joblib")
joblib.dump(frp_model, f"{DRIVE}/model_frp.joblib")
json.dump({"features": FEATURES, "grid_degrees": GRID}, open(f"{DRIVE}/feature_schema.json","w"), indent=2)
json.dump({"presence_threshold": threshold, "presence_model": best_cls[1], "count_model": best_reg[1],
           "frp_model": "lightgbm log1p", "trained_on": TRAIN_YEARS}, open(f"{DRIVE}/model_meta.json","w"), indent=2)
json.dump(results, open(f"{DRIVE}/metrics.json","w"), indent=2)
print("saved to", DRIVE)

## 6) التقدير لسنوات MODIS فقط (2000–2012)

كل صف هنا `is_estimate = True`. الموديل يتعلم من خلايا MODIS فيها حريق، فالتقدير بيغطي
الخلايا اللي MODIS شافها فقط — الـ VIIRS-only cell-days مش قابلة للتقدير بمدخلات MODIS لوحدها.

In [ ]:
def predict_year(modis_raw, year, threshold):
    cd = aggregate(clean(modis_raw, "modis"), "modis")
    table = add_context(cd); table["year"] = year
    Xp = table[FEATURES]
    proba = best_cls[2].predict_proba(Xp)[:,1]
    counts = np.clip(best_reg[2].predict(Xp), 0, None)
    frp = np.clip(np.expm1(frp_model.predict(Xp)), 0, None)
    out = pd.DataFrame({"cell_id": table.cell_id, "date": table.date.dt.strftime("%Y-%m-%d"),
                        "cell_lat": table.cell_lat.round(4), "cell_lon": table.cell_lon.round(4),
                        "modis_detections": table.modis_detections.astype(int),
                        "viirs_like_presence_prob": proba.round(4),
                        "viirs_like_presence": (proba >= threshold).astype(int),
                        "viirs_like_detections": counts.round(2), "viirs_like_frp_mean": frp.round(3),
                        "year": year, "is_estimate": True})
    out.loc[out.viirs_like_presence == 0, ["viirs_like_detections","viirs_like_frp_mean"]] = 0.0
    path = f"{DRIVE}/viirs_like_{year}.csv"
    out.to_csv(path, index=False)
    print(f"{year}: {len(out):,} cell-days, expected VIIRS cell-days {int(out.viirs_like_presence.sum()):,}, "
          f"expected detections {out.viirs_like_detections.sum():,.0f} -> {path}")
    return out

predictions = {y: predict_year(raw[("modis", y)], y, threshold) for y in PREDICT_YEARS}

## ملاحظات علمية لازم تتحط مع أي نتيجة

1. **VIIRS لم يكن موجودًا قبل 2012**؛ المخرجات تقديرات موديل وليست أرصادًا.
2. الداتا المُستخدمة للتدريب هي الخلايا اللي فيها MODIS، يعني الحالات اللي VIIRS شافها وMODIS فاتته
   مش موجودة في التدريب؛ النتيجة إجمالي تقديري أقل من إجمالي VIIRS الحقيقي، ولازم تتحسب نسبة تغطية
   (`joined VIIRS cell-days / all VIIRS cell-days`) وتُذكر مع أي مجموع.
3. الـ split زمني/مجموعة-يومي، والتحقق الأقوى هو تدريب على سنين واختبار على سنة مختلفة تمامًا.
4. FRP الأدق هو log1p(FRP) للخلايا اللي فيها حريق، لأن FRP متوسط خلية-يوم مشتق من خلط بكسلات مختلف الأحجام.
